In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, BatchNormalization, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras import regularizers
# import optuna
# from optuna.integration import TFKerasPruningCallback
import keras_tuner as kt
import warnings
import os

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)


In [5]:
# Define hemorrhage types and custom class weights (for later use)
HEMORRHAGE_TYPES = ['epidural', 'intraparenchymal', 'intraventricular', 'subarachnoid', 'subdural', 'any']
CLASS_WEIGHTS = {
    0: 1.0,  # epidural
    1: 1.0,  # intraparenchymal
    2: 1.0,  # intraventricular
    3: 1.0,  # subarachnoid
    4: 1.0,  # subdural
    5: 2.0   # any (weighted higher)
}


In [6]:
def weighted_binary_crossentropy(y_true, y_pred):
    """
    Custom loss function that applies different weights to different classes.
    The 'any' category (last column) gets a higher weight.
    """
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    # Create a weights tensor: same shape as y_true, with higher weight for 'any' (last column)
    weights = tf.ones_like(y_true)
    batch_size = tf.shape(y_true)[0]
    indices = tf.stack([tf.range(batch_size), tf.fill([batch_size], 5)], axis=1)
    weights = tf.tensor_scatter_nd_update(weights, indices, tf.fill([batch_size], 2.0))
    weighted_loss = bce * weights
    return tf.reduce_mean(weighted_loss)


In [7]:
def create_model(hp):
    """
    Create a Keras model with hyperparameters defined by hp.
    """
    input_dim = hp.Int('input_dim', min_value=128, max_value=1024, step=128, default=512)
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    
    n_blocks = hp.Int('n_blocks', min_value=2, max_value=5, default=3)
    neurons = hp.Int('initial_neurons', min_value=128, max_value=512, step=64, default=256)
    reduction_factor = hp.Float('reduction_factor', min_value=0.5, max_value=0.8, step=0.1, default=0.7)
    dropout_rate = hp.Float('dropout_rate', min_value=0.2, max_value=0.5, step=0.1, default=0.3)
    l2_reg = hp.Float('l2_reg', min_value=1e-6, max_value=1e-3, sampling='log', default=1e-4)
    
    for i in range(n_blocks):
        neurons_i = int(neurons * (reduction_factor ** i))
        model.add(Dense(neurons_i, 
                        kernel_regularizer=regularizers.l2(l2_reg),
                        kernel_initializer='he_normal'))
        model.add(BatchNormalization())
        activation = hp.Choice(f'activation_{i}', ['relu', 'elu', 'selu'], default='relu')
        model.add(Activation(activation))
        model.add(Dropout(dropout_rate))
    
    model.add(Dense(len(HEMORRHAGE_TYPES), activation='sigmoid'))
    
    learning_rate = hp.Float('learning_rate', min_value=1e-5, max_value=1e-2, sampling='log', default=1e-3)
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss=weighted_binary_crossentropy,
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return model


In [8]:
def create_simple_model(input_dim, n_blocks=3, initial_neurons=256, reduction_factor=0.5, 
                        dropout_rate=0.3, l2_reg=1e-4, learning_rate=1e-3):
    """
    Create a simple Keras model with fixed hyperparameters.
    """
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    
    for i in range(n_blocks):
        neurons_i = int(initial_neurons * (reduction_factor ** i))
        model.add(Dense(neurons_i, 
                        kernel_regularizer=regularizers.l2(l2_reg), 
                        kernel_initializer='he_normal'))
        model.add(BatchNormalization())
        model.add(Activation('relu'))
        model.add(Dropout(dropout_rate))
    
    model.add(Dense(len(HEMORRHAGE_TYPES), activation='sigmoid'))
    
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss=weighted_binary_crossentropy,
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return model


In [11]:

# Load the datasets
df_hemo = pd.read_csv(os.path.expanduser('~/merged_hemorrhage_embeddings.csv'))
df_control = pd.read_csv(os.path.expanduser('~/merged_control_embeddings.csv'))

# Combine the datasets
df = pd.concat([df_hemo, df_control], ignore_index=True)

# Convert the 'embedding' column from string to a list
df['embedding'] = df['embedding'].apply(eval)

# Define the target columns in the proper order
target_columns = ['any', 'epidural', 'intraparenchymal', 'intraventricular', 'subarachnoid', 'subdural']

# Extract features and labels
X = np.array(df['embedding'].tolist(), dtype=np.float32)
y = df[target_columns].values.astype(np.float32)

# Split the data (20% for testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("y_train shape:", y_train.shape)


X_train_scaled shape: (12900, 1408)
y_train shape: (12900, 6)


In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, batch_size=32, epochs=100, patience=10):
    """
    Train the model with early stopping, learning rate reduction, and checkpointing.
    """
    callbacks = [
        EarlyStopping(monitor='val_auc', patience=patience, mode='max', restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=5, min_lr=1e-6, mode='max', verbose=1),
        ModelCheckpoint('best_model.h5', monitor='val_auc', mode='max', save_best_only=True, verbose=1)
    ]
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=batch_size,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )
    return history


In [ ]:
def evaluate_model(model, X_test, y_test):
    """
    Evaluate the model on test data and print AUC scores and classification reports.
    """
    y_pred = model.predict(X_test)
    auc_scores = {}
    for i, hemorrhage_type in enumerate(HEMORRHAGE_TYPES):
        auc = roc_auc_score(y_test[:, i], y_pred[:, i])
        auc_scores[hemorrhage_type] = auc
        print(f"AUC for {hemorrhage_type}: {auc:.4f}")
    
    overall_auc = np.mean(list(auc_scores.values()))
    print(f"Overall AUC: {overall_auc:.4f}")
    
    y_pred_binary = (y_pred > 0.5).astype(int)
    for i, hemorrhage_type in enumerate(HEMORRHAGE_TYPES):
        print(f"\nClassification Report for {hemorrhage_type}:")
        print(classification_report(y_test[:, i], y_pred_binary[:, i]))
    
    return {'auc_scores': auc_scores, 'overall_auc': overall_auc, 'y_pred': y_pred, 'y_pred_binary': y_pred_binary}


In [ ]:
# Function to perform hyperparameter tuning with Keras Tuner
def tune_hyperparameters_keras_tuner(X_train, y_train, X_val, y_val, max_trials=50):
    """
    Perform hyperparameter tuning using Keras Tuner.
    
    Args:
        X_train, y_train: Training data
        X_val, y_val: Validation data
        max_trials: Maximum number of trials
        
    Returns:
        Best hyperparameters and best model
    """
    # Set input dimension
    input_dim = X_train.shape[1]
    
    # Create tuner
    tuner = kt.Hyperband(
        create_model,
        objective=kt.Objective('val_auc', direction='max'),
        max_epochs=50,
        factor=3,
        directory='keras_tuner',
        project_name='hemorrhage_classification',
        overwrite=True
    )
    
    # Define early stopping
    stop_early = tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', 
        mode='max',
        patience=5
    )
    
    # Search for best hyperparameters
    tuner.search(
        X_train, y_train,
        epochs=50,
        validation_data=(X_val, y_val),
        callbacks=[stop_early],
        verbose=1
    )
    
    # Get best hyperparameters
    best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
    print("Best hyperparameters:")
    for hp in best_hps.values:
        print(f"  {hp}: {best_hps.values[hp]}")
    
    # Build model with best hyperparameters
    best_model = tuner.hypermodel.build(best_hps)
    
    return best_hps, best_model